# Hermes Reasoning Tool Use — Dataset Analysis & Masking

This notebook:
1. Loads `hermes_reasoning_tool_use_train.jsonl` and profiles turn counts, role patterns, and scenario categories.
2. Shows how the `ToolAwareCompletionsCollator` masks a few real examples from the dataset.

In [ ]:
import json
from collections import Counter

from IPython.display import HTML, display
from transformers import AutoTokenizer

from oumi.core.collators.tool_aware_completions_collator import (
    ToolAwareCompletionsCollator,
)

DATA_PATH = "/data/shanghong/oumi/gold/data/hermes_reasoning_tool_use_train.jsonl"
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

## 1. Load dataset

In [ ]:
records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))

print(f"Total examples: {len(records):,}")

## 2. Turn count distribution

In [ ]:
turn_counts = Counter(len(r["messages"]) for r in records)

print(f"{'Turns':>6}  {'Count':>7}  {'%':>6}")
print("-" * 24)
for n_turns in sorted(turn_counts):
    count = turn_counts[n_turns]
    print(f"{n_turns:>6}  {count:>7,}  {100 * count / len(records):>5.1f}%")

## 3. Scenario categories

In [ ]:
categories = Counter(r["metadata"].get("scenario_category", "unknown") for r in records)

print(f"{'Category':<15}  {'Count':>7}  {'%':>6}")
print("-" * 32)
for cat, count in categories.most_common():
    print(f"{cat:<15}  {count:>7,}  {100 * count / len(records):>5.1f}%")

## 4. Role patterns

In [ ]:
role_patterns = Counter(tuple(m["role"] for m in r["messages"]) for r in records)

print(f"Unique role patterns: {len(role_patterns)}\n")
print(f"{'Count':>7}  Role sequence")
print("-" * 70)
for roles, count in role_patterns.most_common(10):
    print(f"{count:>7,}  {' -> '.join(roles)}")

## 5. Role frequency per position

Which roles appear at each turn index across the dataset?

In [ ]:
max_turns = max(len(r["messages"]) for r in records)
role_at_position = [Counter() for _ in range(max_turns)]
for r in records:
    for i, m in enumerate(r["messages"]):
        role_at_position[i][m["role"]] += 1

print(f"{'Pos':>4}  {'system':>7}  {'user':>7}  {'assistant':>10}  {'tool':>7}")
print("-" * 42)
for i, counter in enumerate(role_at_position):
    if sum(counter.values()) == 0:
        break
    print(
        f"{i:>4}  {counter.get('system', 0):>7,}  {counter.get('user', 0):>7,}  "
        f"{counter.get('assistant', 0):>10,}  {counter.get('tool', 0):>7,}"
    )

---
## 6. Masking analysis with `ToolAwareCompletionsCollator`

Load a tokenizer and the new collator, then visualise masking on real examples.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
END_OF_TURN = "<|im_end|>"

collator = ToolAwareCompletionsCollator(
    response_template=RESPONSE_TEMPLATE,
    end_of_turn_template=END_OF_TURN,
    mask_tool_calls=False,  # train on tool call generation
    tokenizer=tokenizer,
)

print("Collator ready (mask_tool_calls=False)")

In [ ]:
def show_masking(input_ids, labels, tokenizer):
    """Render color-coded token masking. Green = trains, Red = masked."""
    masked_style = (
        "background:#ffcccc;border:1px solid #ff9999;"
        "border-radius:3px;padding:1px 2px;margin:1px;"
    )
    unmasked_style = (
        "background:#ccffcc;border:1px solid #66cc66;"
        "border-radius:3px;padding:1px 2px;margin:1px;"
    )
    parts = []
    for tok_id, lbl in zip(input_ids, labels):
        text = tokenizer.decode([tok_id])
        text = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        text = text.replace(" ", "&nbsp;").replace("\n", "\u21b5<br/>")
        if lbl == -100:
            style, title = masked_style, "masked"
        else:
            style, title = unmasked_style, f"label={lbl}"
        parts.append(f'<span style="{style}" title="{title}">{text}</span>')
    display(
        HTML(
            '<div style="font-family:monospace;line-height:2;max-height:600px;'
            'overflow-y:auto;border:1px solid #ccc;padding:8px;">'
            + "".join(parts)
            + "</div>"
        )
    )


def analyse_example(record, collator, tokenizer, max_tokens=1024):
    """Tokenize, collate, and display masking for one dataset record."""
    messages = record["messages"]
    roles = [m["role"] for m in messages]
    category = record["metadata"].get("scenario_category", "unknown")
    conv_id = record.get("conversation_id", "?")

    # Render conversation through the chat template
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    token_ids = tokenizer.encode(rendered, add_special_tokens=False)

    # Truncate for display
    truncated = len(token_ids) > max_tokens
    token_ids_display = token_ids[:max_tokens]

    # Run collator
    batch = collator.torch_call([{"input_ids": token_ids_display}])
    input_ids = batch["input_ids"][0].tolist()
    labels = batch["labels"][0].tolist()

    total = len(labels)
    unmasked = sum(1 for tok in labels if tok != -100)
    masked = total - unmasked

    # Header
    print(f"=== Conversation {conv_id} | category: {category} ===")
    print(f"Turns: {len(messages)}  |  Roles: {' -> '.join(roles)}")
    trunc_msg = f" (truncated to {max_tokens})" if truncated else ""
    print(f"Tokens: {len(token_ids)}{trunc_msg}")
    unmasked_pct = 100 * unmasked / total
    masked_pct = 100 * masked / total
    print(
        f"Unmasked: {unmasked}/{total} ({unmasked_pct:.1f}%)"
        f"  |  Masked: {masked}/{total} ({masked_pct:.1f}%)"
    )
    print()

    # Per-role token breakdown
    return input_ids, labels

### Example 1: `single` — no tool result turn (system + user + assistant)

The most common pattern (21k examples). Only 3 turns, no tool result. The assistant response contains `<tool_call>` but no tool result follows.

In [ ]:
# Find first example with pattern: system -> user -> assistant
ex_single = next(
    r
    for r in records
    if tuple(m["role"] for m in r["messages"]) == ("system", "user", "assistant")
)
ids, labels = analyse_example(ex_single, collator, tokenizer)
show_masking(ids, labels, tokenizer)

### Example 2: `single` — with tool result (system + user + assistant + tool)

Second most common pattern (7k examples). The assistant generates a tool call, and a tool result follows. The tool result should be **masked** (red).

In [ ]:
ex_with_tool = next(
    r
    for r in records
    if tuple(m["role"] for m in r["messages"])
    == ("system", "user", "assistant", "tool")
)
ids, labels = analyse_example(ex_with_tool, collator, tokenizer, max_tokens=4096)
show_masking(ids, labels, tokenizer)

### Example 3: `single` — assistant calls tool, gets result, then answers

5-turn pattern (4.3k examples): system -> user -> assistant(tool_call) -> tool(result) -> assistant(answer).  
Both assistant turns should be **green**. Tool result should be **red**.

In [ ]:
ex_5turn = next(
    r
    for r in records
    if tuple(m["role"] for m in r["messages"])
    == ("system", "user", "assistant", "tool", "assistant")
)
ids, labels = analyse_example(ex_5turn, collator, tokenizer, max_tokens=4096)
show_masking(ids, labels, tokenizer)

### Example 4: `multiturn` — multiple user/assistant/tool rounds

9-turn pattern (11k examples): system -> user -> assistant -> tool -> assistant -> user -> assistant -> tool -> assistant.  
All assistant turns should be **green**. All tool/user turns should be **red**.

In [ ]:
ex_multiturn = next(
    r
    for r in records
    if tuple(m["role"] for m in r["messages"])
    == (
        "system",
        "user",
        "assistant",
        "tool",
        "assistant",
        "user",
        "assistant",
        "tool",
        "assistant",
    )
)
ids, labels = analyse_example(ex_multiturn, collator, tokenizer, max_tokens=4096)
show_masking(ids, labels, tokenizer)

### Example 5: `multistep` — assistant calls multiple tools in sequence

8-turn pattern (3k examples): system -> user -> assistant -> tool -> assistant -> tool -> assistant -> assistant.  
Note the back-to-back tool calls and the consecutive assistant turns at the end.

In [ ]:
ex_multistep = next(
    r
    for r in records
    if tuple(m["role"] for m in r["messages"])
    == (
        "system",
        "user",
        "assistant",
        "tool",
        "assistant",
        "tool",
        "assistant",
        "assistant",
    )
)
ids, labels = analyse_example(ex_multistep, collator, tokenizer, max_tokens=4096)
show_masking(ids, labels, tokenizer)

---
## 7. Aggregate masking statistics

Run the collator over a sample of examples and measure what fraction of tokens are trained on vs masked, broken down by scenario category.

In [ ]:
import random

random.seed(42)
sample = random.sample(records, min(500, len(records)))

stats_by_cat = {}  # category -> {total_tokens, unmasked_tokens, count}

for rec in sample:
    cat = rec["metadata"].get("scenario_category", "unknown")
    if cat not in stats_by_cat:
        stats_by_cat[cat] = {"total": 0, "unmasked": 0, "count": 0}

    rendered = tokenizer.apply_chat_template(
        rec["messages"], tokenize=False, add_generation_prompt=False
    )
    token_ids = tokenizer.encode(rendered, add_special_tokens=False)

    batch = collator.torch_call([{"input_ids": token_ids}])
    labels = batch["labels"][0].tolist()

    n_total = len(labels)
    n_unmasked = sum(1 for tok in labels if tok != -100)

    stats_by_cat[cat]["total"] += n_total
    stats_by_cat[cat]["unmasked"] += n_unmasked
    stats_by_cat[cat]["count"] += 1

header = (
    f"{'Category':<15}  {'Samples':>8}  {'Avg tokens':>11}"
    f"  {'Avg unmasked':>13}  {'% trained':>10}"
)
print(header)
print("-" * 65)
total_all, unmasked_all = 0, 0
for cat in sorted(stats_by_cat):
    s = stats_by_cat[cat]
    avg_tok = s["total"] / s["count"]
    avg_unmask = s["unmasked"] / s["count"]
    pct = 100 * s["unmasked"] / s["total"] if s["total"] > 0 else 0
    print(
        f"{cat:<15}  {s['count']:>8}"
        f"  {avg_tok:>11.0f}  {avg_unmask:>13.0f}  {pct:>9.1f}%"
    )
    total_all += s["total"]
    unmasked_all += s["unmasked"]

print("-" * 65)
avg_tok_all = total_all / len(sample)
avg_unmask_all = unmasked_all / len(sample)
pct_all = 100 * unmasked_all / total_all
print(
    f"{'OVERALL':<15}  {len(sample):>8}  {avg_tok_all:>11.0f}"
    f"  {avg_unmask_all:>13.0f}  {pct_all:>9.1f}%"
)